# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.1 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile, glob, sys,math, random, collections, csv
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict


In [5]:
TASK_ID = "task033"
CH = 10
H = W = 30
MAX_ONNX_BYTES = 1_400_000
FORBIDDEN_OPS = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
ONNX_PATH = WORKDIR / f"{TASK_ID}_static_graph.onnx"
ZIP_PATH = WORKDIR / f"{TASK_ID}_static_graph_submission.zip"
GENERIC_ZIP = WORKDIR / "submission.zip"
HEALTH_PATH = WORKDIR / f"{TASK_ID}_verified_onnx_health.json"

In [6]:
def find_task_json(task_id: str) -> Path:
    candidates = [
        Path.cwd() / f"{task_id}.json",
        Path("/mnt/data") / f"{task_id}.json",
        Path("/kaggle/working") / f"{task_id}.json",
    ]
    for p in candidates:
        if p.exists():
            return p
    for root in [Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            matches = list(root.rglob(f"{task_id}.json"))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Could not find {task_id}.json")

TASK_PATH = find_task_json(TASK_ID)
with open(TASK_PATH) as f:
    task = json.load(f)
print("Loaded", TASK_PATH)
print({k: len(task.get(k, [])) for k in ["train", "test", "arc-gen"]})

Loaded /kaggle/input/competitions/neurogolf-2026/task033.json
{'train': 3, 'test': 1, 'arc-gen': 261}


In [7]:
def encode_grid(grid, H: int = 30, W: int = 30) -> np.ndarray:
    arr = np.array(grid, dtype=np.int64)
    x = np.zeros((1, 10, H, W), dtype=np.float32)
    h, w = arr.shape
    assert h <= H and w <= W
    for c in range(10):
        x[0, c, :h, :w] = (arr == c).astype(np.float32)
    return x


def decode_onehot(y: np.ndarray, h: int, w: int) -> np.ndarray:
    if y.ndim == 4:
        y = y[0]
    return y[:, :h, :w].argmax(axis=0).astype(np.int64)


def grid_shape(ex):
    return len(ex["input"]), len(ex["input"][0])

In [8]:
class Task033UnionStampModel(nn.Module):
    def __init__(self):
        super().__init__()
        basis = torch.zeros(1, 9, 30, 30, dtype=torch.float32)
        k = 0
        for ro in [1, 2, 3]:
            for co in [1, 2, 3]:
                for br in [0, 6, 12]:
                    for bc in [0, 6, 12]:
                        basis[0, k, br + ro, bc + co] = 1.0
                k += 1
        nonbg = torch.ones(1, 10, 1, 1, dtype=torch.float32)
        nonbg[:, 0] = 0.0
        self.register_buffer("basis", basis)
        self.register_buffer("nonbg", nonbg)

    def forward(self, x):
        counts = x[:, 1:, :, :].sum(dim=(2, 3), keepdim=True)
        max_count = counts.amax(dim=1, keepdim=True)
        zero = torch.zeros_like(max_count)
        counts_full = torch.cat([zero, counts], dim=1)

        divider = ((torch.abs(counts_full - max_count) < 0.1) & (counts_full > 0.5)).to(x.dtype)
        non_divider_non_bg = (1.0 - divider) * self.nonbg.to(x.dtype)
        object_any = (x * non_divider_non_bg).sum(dim=1, keepdim=True)

        union = ((object_any * self.basis.to(x.dtype)).sum(dim=(2, 3), keepdim=True) > 0.5).to(x.dtype)
        stamp = (self.basis.to(x.dtype) * union).sum(dim=1, keepdim=True)

        fill_background = stamp * x[:, 0:1, :, :]
        return x * (1.0 - fill_background) + divider * fill_background


model = Task033UnionStampModel().eval()

In [9]:
def run_torch(ex):
    h, w = grid_shape(ex)
    x = torch.tensor(encode_grid(ex["input"]), dtype=torch.float32)
    with torch.no_grad():
        y = model(x).cpu().numpy()
    return decode_onehot(y, h, w)

for split in ["train", "test", "arc-gen"]:
    ok = 0
    bad = []
    for i, ex in enumerate(task.get(split, [])):
        pred = run_torch(ex)
        gold = np.array(ex["output"], dtype=np.int64)
        good = np.array_equal(pred, gold)
        ok += int(good)
        if not good and len(bad) < 5:
            bad.append(i)
    print(split, ok, "/", len(task.get(split, [])), "bad", bad)
    assert ok == len(task.get(split, []))

train 3 / 3 bad []
test 1 / 1 bad []
arc-gen 261 / 261 bad []


In [10]:
dummy = torch.zeros(1, CH, H, W, dtype=torch.float32)
dummy[:, 0, :17, :17] = 1.0
# Build a valid 17×17 lattice example for tracing.
for idx in [5, 11]:
    dummy[:, 0, idx, :17] = 0.0
    dummy[:, 8, idx, :17] = 1.0
    dummy[:, 0, :17, idx] = 0.0
    dummy[:, 8, :17, idx] = 1.0
dummy[:, 0, 1, 2] = 0.0
dummy[:, 4, 1, 2] = 1.0

try:
    torch.onnx.export(
        model,
        dummy,
        str(ONNX_PATH),
        input_names=["input"],
        output_names=["output"],
        opset_version=17,
        do_constant_folding=True,
        dynamic_axes=None,
        dynamo=False,
    )
except TypeError:
    torch.onnx.export(
        model,
        dummy,
        str(ONNX_PATH),
        input_names=["input"],
        output_names=["output"],
        opset_version=17,
        do_constant_folding=True,
        dynamic_axes=None,
    )
print("Wrote", ONNX_PATH, "size", ONNX_PATH.stat().st_size)

/tmp/ipykernel_15/3769903008.py:13: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Wrote /kaggle/working/task033_static_graph.onnx size 35877


In [11]:
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
ops = sorted({node.op_type for node in onnx_model.graph.node})
forbidden_found = sorted(FORBIDDEN_OPS.intersection(ops))
input_shape = [d.dim_value for d in onnx_model.graph.input[0].type.tensor_type.shape.dim]
output_shape = [d.dim_value for d in onnx_model.graph.output[0].type.tensor_type.shape.dim]
size_bytes = ONNX_PATH.stat().st_size

print("input_shape", input_shape)
print("output_shape", output_shape)
print("size_bytes", size_bytes)
print("forbidden_found", forbidden_found)
print("ops", ops)

assert input_shape == [1, 10, 30, 30]
assert output_shape == [1, 10, 30, 30]
assert size_bytes < MAX_ONNX_BYTES
assert not forbidden_found

input_shape [1, 10, 30, 30]
output_shape [1, 10, 30, 30]
size_bytes 35877
forbidden_found []
ops ['Abs', 'Add', 'And', 'Cast', 'Concat', 'Constant', 'Greater', 'Identity', 'Less', 'Mul', 'ReduceMax', 'ReduceSum', 'Slice', 'Sub']


In [12]:
sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])

def run_onnx(ex):
    h, w = grid_shape(ex)
    y = sess.run(None, {"input": encode_grid(ex["input"])})[0]
    return decode_onehot(y, h, w)

summary = {}
for split in ["train", "test", "arc-gen"]:
    ok = 0
    bad = []
    for i, ex in enumerate(task.get(split, [])):
        pred = run_onnx(ex)
        gold = np.array(ex["output"], dtype=np.int64)
        good = np.array_equal(pred, gold)
        ok += int(good)
        if not good and len(bad) < 5:
            bad.append(i)
    summary[split] = {"ok": ok, "total": len(task.get(split, [])), "bad_examples": bad}
    print(split, summary[split])

arc_gen = task.get("arc-gen", [])
cut = int(len(arc_gen) * 0.40)
for name, subset in [("arc_gen_fit_40_percent", arc_gen[:cut]), ("arc_gen_holdout_60_percent", arc_gen[cut:])]:
    ok = 0
    bad = []
    for i, ex in enumerate(subset):
        pred = run_onnx(ex)
        gold = np.array(ex["output"], dtype=np.int64)
        good = np.array_equal(pred, gold)
        ok += int(good)
        if not good and len(bad) < 5:
            bad.append(i)
    summary[name] = {"ok": ok, "total": len(subset), "bad_examples": bad}
    print(name, summary[name])

assert summary["train"]["ok"] == summary["train"]["total"]
assert summary["test"]["ok"] == summary["test"]["total"]
assert summary["arc_gen_fit_40_percent"]["ok"] == summary["arc_gen_fit_40_percent"]["total"]
assert summary["arc_gen_holdout_60_percent"]["ok"] == summary["arc_gen_holdout_60_percent"]["total"]

train {'ok': 3, 'total': 3, 'bad_examples': []}
test {'ok': 1, 'total': 1, 'bad_examples': []}
arc-gen {'ok': 261, 'total': 261, 'bad_examples': []}
arc_gen_fit_40_percent {'ok': 104, 'total': 104, 'bad_examples': []}
arc_gen_holdout_60_percent {'ok': 157, 'total': 157, 'bad_examples': []}


In [13]:
health = {
    "task_id": TASK_ID,
    "model_type": "static neural-symbolic PyTorch tensor model exported to ONNX",
    "not_lookup": True,
    "uses_tree_based_method": False,
    "exact_input_output_bank": False,
    "visible_train": summary["train"],
    "visible_test": summary["test"],
    "arc_gen_all": summary["arc-gen"],
    "arc_gen_fit_40_percent": summary["arc_gen_fit_40_percent"],
    "arc_gen_holdout_60_percent": summary["arc_gen_holdout_60_percent"],
    "onnx_size_bytes": size_bytes,
    "onnx_input_shape": input_shape,
    "onnx_output_shape": output_shape,
    "forbidden_ops_found": forbidden_found,
    "onnx_ops": ops,
}
health["passes_required_checks"] = bool(
    summary["train"]["ok"] == summary["train"]["total"]
    and summary["test"]["ok"] == summary["test"]["total"]
    and summary["arc_gen_fit_40_percent"]["ok"] == summary["arc_gen_fit_40_percent"]["total"]
    and summary["arc_gen_holdout_60_percent"]["ok"] == summary["arc_gen_holdout_60_percent"]["total"]
    and size_bytes < MAX_ONNX_BYTES
    and not forbidden_found
    and input_shape == [1, 10, 30, 30]
    and output_shape == [1, 10, 30, 30]
)
with open(HEALTH_PATH, "w") as f:
    json.dump(health, f, indent=2)

for zip_path in [ZIP_PATH, GENERIC_ZIP]:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")
print("Wrote", ZIP_PATH)
print("Wrote", GENERIC_ZIP)
print(json.dumps(health, indent=2))
assert health["passes_required_checks"]

Wrote /kaggle/working/task033_static_graph_submission.zip
Wrote /kaggle/working/submission.zip
{
  "task_id": "task033",
  "model_type": "static neural-symbolic PyTorch tensor model exported to ONNX",
  "not_lookup": true,
  "uses_tree_based_method": false,
  "exact_input_output_bank": false,
  "visible_train": {
    "ok": 3,
    "total": 3,
    "bad_examples": []
  },
  "visible_test": {
    "ok": 1,
    "total": 1,
    "bad_examples": []
  },
  "arc_gen_all": {
    "ok": 261,
    "total": 261,
    "bad_examples": []
  },
  "arc_gen_fit_40_percent": {
    "ok": 104,
    "total": 104,
    "bad_examples": []
  },
  "arc_gen_holdout_60_percent": {
    "ok": 157,
    "total": 157,
    "bad_examples": []
  },
  "onnx_size_bytes": 35877,
  "onnx_input_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_output_shape": [
    1,
    10,
    30,
    30
  ],
  "forbidden_ops_found": [],
  "onnx_ops": [
    "Abs",
    "Add",
    "And",
    "Cast",
    "Concat",
    "Constant",
    "Greater",
   